# 04 Global vs specialized models

One LightGBM model for every series, or separate models per group? And does the answer depend on how we scale the target?

We keep the model family, features and parameters fixed and change one thing at a time:

- **pooling**: all series in one model, one model per department, one model per sales-volume tier
- **target scaling**: raw units, per-series standardization, per-series min-max

Local statistical forecasts from notebook 01 are the reference. At the end we shorten some products' history to see how each approach copes.

Optional in the live session. The main part runs in about two minutes; the optional partition experiment at the end takes longer.


In [1]:
import os
import sys
import warnings

if "google.colab" in sys.modules:
    !git clone -q https://github.com/tackes/time-series-forecasting-bootcamp
    %cd time-series-forecasting-bootcamp
    %pip install -q statsforecast==2.1.1 mlforecast==1.1.0 utilsforecast==0.2.17 lightgbm==4.7.0 catboost==1.2.10
elif not os.path.exists("shared"):
    os.chdir("..")
sys.path.insert(0, os.getcwd())
warnings.filterwarnings("ignore")

In [2]:
import numpy as np
import pandas as pd
import lightgbm as lgb
from mlforecast import MLForecast
from mlforecast.target_transforms import LocalStandardScaler, LocalMinMaxScaler
from statsforecast import StatsForecast
from statsforecast.models import HistoricAverage

from shared.prepare_data import load_panel, series_segments, H, FREQ, ALL_CUTOFFS, DEV_CUTOFFS, TEST_CUTOFF
from shared.features import LAGS, LAG_TRANSFORMS, DATE_FEATURES, LGB_PARAMS
from shared.evaluation import score
from shared.leaderboard import to_long, save_forecasts, load_forecasts, leaderboard, breakdown

panel = load_panel()
actuals = panel[["unique_id", "ds", "y"]]
segments = series_segments(panel)

## Five ways to use product groups

These are different operations:

| approach | what is trained | how the groups enter | where |
|---|---|---|---|
| one global model | one LightGBM on all 2,000 series | not at all | `lgbm_baseline` (02) |
| partitioned models | one LightGBM per department, class or subclass | each model sees only its group's series | this notebook |
| global model with categorical attributes | one LightGBM on all series | department, class and subclass as `category` features the trees can split on | `lgbm_l1_std_categorical` (03) |
| behavioral clustering | either of the above | groups formed from demand behavior (volume, intermittency, seasonality) instead of the merchandise hierarchy | the volume tiers below are a simple version; no clustering model here |
| hierarchical model selection | nothing new | groups decide which already-trained model's forecast each series uses | 07 |

Partitioning changes which data each model learns from. Categorical attributes keep one model and let it learn how groups differ. Selection happens after training and changes no model.

In the partition experiment at the end, a group gets its own model only if it has at least `MIN_SERIES = 10` series at the origin; the other series use the global model. At the final origin that specializes 87% of series by department, 50% by class and 15% by subclass.

## Groups

Departments with fewer than 30 series in the sample are combined into "Other", and products without a department form "Unknown". Group membership is fixed per product, so it carries no future information.

In [3]:
group = panel.drop_duplicates("unique_id").set_index("unique_id").department_name.fillna("Unknown")
group = group.where(group.map(group.value_counts()) >= 30, "Other")
group.value_counts()

department_name
Other                 1009
Cooking Needs          100
Stationery              79
Condiments              79
Confectionery           65
Baby                    52
Seasonal                49
Household Cleaners      46
Soft Drinks             45
Packaged                44
Medicinal               44
Ice Cream               43
International Food      41
Snacks                  37
Dog Food                35
Beverages               35
Health Food             34
Electrical              34
Delicatessen            34
Oral Care               33
Coffee                  32
Pet Accessories         30
Name: count, dtype: int64

Volume tiers are different: they are recomputed at every cutoff from the mean sales in the training window, so a product can change tier between origins.

In [4]:
def volume_tiers(history):
    mean_sales = history.groupby("unique_id").y.mean()
    return pd.qcut(mean_sales.rank(method="first"), 3, labels=["low", "mid", "high"]).astype(str)

## One function, three pooling strategies

`fit_groups` trains one MLForecast model per group and stacks the forecasts. A single group containing everyone is the global model. Everything else is identical.

In [5]:
def make_model(name, target_transforms):
    return MLForecast(
        models={name: lgb.LGBMRegressor(**LGB_PARAMS)},
        freq=FREQ,
        lags=LAGS,
        lag_transforms=LAG_TRANSFORMS,
        date_features=DATE_FEATURES,
        target_transforms=target_transforms,
        num_threads=1,
    )

def fit_groups(history, membership, name, target_transforms):
    forecasts = []
    for _, ids in membership.groupby(membership).groups.items():
        mlf = make_model(name, target_transforms)
        mlf.fit(history[history.unique_id.isin(ids)])
        forecasts.append(mlf.predict(H))
    return pd.concat(forecasts)

In [6]:
forecasts = []
for cutoff in ALL_CUTOFFS:
    history = actuals[actuals.ds <= cutoff]
    everyone = pd.Series("all", index=history.unique_id.unique())
    tiers = volume_tiers(history)

    runs = {
        "lgbm_global_minmax": (everyone, [LocalMinMaxScaler()]),
        "lgbm_dept_raw": (group, []),
        "lgbm_dept_std": (group, [LocalStandardScaler()]),
        "lgbm_velocity_std": (tiers, [LocalStandardScaler()]),
    }
    for name, (membership, transforms) in runs.items():
        forecasts.append(to_long(fit_groups(history, membership, name, transforms), cutoff, "04_global_vs_specialized"))

forecasts = pd.concat(forecasts)

`lgbm_raw` (global, raw target) and `lgbm_baseline` (global, standardized) come from notebook 02 and are the same code run on the same data, so they are loaded instead of refit.

In [7]:
reference = load_forecasts(["01_baselines", "02_ml_baseline"])
reference = reference[reference.model.isin(["Naive", "HistoricAverage", "SeasonalNaive", "lgbm_raw", "lgbm_baseline"])]
candidates = pd.concat([reference, forecasts])

columns = ["model", "wape", "bias", "mean_rmsse", "median_rmsse", "apc", "apc_median"]
leaderboard(candidates, actuals, scope="dev")[columns].round(3)

,model,wape,bias,mean_rmsse,median_rmsse,apc,apc_median
0,SeasonalNaive,0.408,-0.042,1.850,1.055,0.000,0.000
1,HistoricAverage,0.426,-0.060,1.791,0.887,0.015,0.016
2,lgbm_dept_std,0.440,0.119,1.865,0.918,0.157,0.153
3,lgbm_velocity_std,0.443,0.134,1.861,0.924,0.130,0.141
4,lgbm_baseline,0.449,0.139,1.865,0.921,0.120,0.127
5,Naive,0.454,-0.088,1.860,0.969,0.352,0.370
6,lgbm_raw,0.455,0.044,5.501,1.224,0.203,0.223
7,lgbm_dept_raw,0.466,0.037,3.867,1.103,0.224,0.256
8,lgbm_global_minmax,0.528,0.249,2.075,0.969,0.152,0.149


Reading the table as an experiment:

- **Pooling** (same scaling): `lgbm_baseline` vs `lgbm_dept_std` vs `lgbm_velocity_std`, and `lgbm_raw` vs `lgbm_dept_raw`.
- **Scaling** (same pooling): `lgbm_raw` vs `lgbm_baseline` vs `lgbm_global_minmax`.

Department models get fewer training rows each; the trade is specialization against data per model.

In [8]:
leaderboard(candidates, actuals, scope="test")[columns].round(3)

,model,wape,bias,mean_rmsse,median_rmsse,apc,apc_median
0,lgbm_dept_std,0.403,0.079,1.312,0.737,0.133,0.111
1,lgbm_baseline,0.419,0.109,1.302,0.733,0.092,0.076
2,HistoricAverage,0.426,-0.011,1.367,0.739,0.010,0.010
3,lgbm_velocity_std,0.432,0.137,1.314,0.742,0.102,0.082
4,SeasonalNaive,0.438,-0.037,1.525,0.929,0.000,0.000
5,lgbm_dept_raw,0.470,0.160,5.775,0.883,0.188,0.195
6,Naive,0.470,0.066,1.332,0.771,0.327,0.308
7,lgbm_raw,0.505,0.215,12.000,1.065,0.164,0.169
8,lgbm_global_minmax,0.520,0.269,1.363,0.830,0.119,0.104


### Reading the result

- Pooling is a small effect. Department models (0.440) and volume-tier models (0.443) are slightly better than the global model (0.449) on development origins; the department models stay ahead in the final window (0.403 against 0.419).
- Scaling is a larger one. Min-max scaling is the worst LightGBM (0.528, bias +0.249) and fails most on zero-heavy products (WAPE 2.17). Department models on the raw target (0.466) are worse than the global raw model (0.455).
- No LightGBM variant here beats seasonal naive (0.408) on development WAPE. Pooling and scaling alone don't fix the baseline's overforecasting.

## By segment

Volume and intermittency are computed from each product's history up to the final cutoff. Development origins only.

In [9]:
by_volume = breakdown(candidates, actuals, "volume", scope="dev", segments=segments)
by_volume.pivot(index="model", columns="volume", values="wape")[["low", "mid", "high"]].round(3)

volume,low,mid,high
model,,,
HistoricAverage,0.694,0.510,0.417
Naive,0.712,0.513,0.447
SeasonalNaive,0.797,0.567,0.391
lgbm_baseline,0.737,0.543,0.439
lgbm_dept_raw,1.832,0.543,0.449
lgbm_dept_std,0.722,0.533,0.430
lgbm_global_minmax,0.848,0.619,0.518
lgbm_raw,2.678,0.507,0.434
lgbm_velocity_std,0.739,0.536,0.433


In [10]:
by_zeros = breakdown(candidates, actuals, "intermittency", scope="dev", segments=segments)
by_zeros.pivot(index="model", columns="intermittency", values="wape")[["regular", "some_zeros", "zero_heavy"]].round(3)

intermittency,regular,some_zeros,zero_heavy
model,,,
HistoricAverage,0.339,0.597,1.030
Naive,0.392,0.587,0.724
SeasonalNaive,0.300,0.623,1.073
lgbm_baseline,0.371,0.580,1.393
lgbm_dept_raw,0.393,0.599,1.175
lgbm_dept_std,0.359,0.581,1.321
lgbm_global_minmax,0.435,0.654,2.168
lgbm_raw,0.382,0.579,1.332
lgbm_velocity_std,0.367,0.579,1.200


In [11]:
by_zeros.pivot(index="model", columns="intermittency", values="bias")[["regular", "some_zeros", "zero_heavy"]].round(3)

intermittency,regular,some_zeros,zero_heavy
model,,,
HistoricAverage,-0.060,-0.023,-0.703
Naive,-0.087,-0.076,-0.313
SeasonalNaive,-0.040,-0.001,-0.815
lgbm_baseline,0.109,0.225,-0.099
lgbm_dept_raw,-0.003,0.120,0.261
lgbm_dept_std,0.083,0.216,-0.068
lgbm_global_minmax,0.202,0.301,1.280
lgbm_raw,-0.002,0.129,0.450
lgbm_velocity_std,0.105,0.230,-0.297


WAPE here is dominated by whichever products sell the most in each segment, so the bias table matters too: a model can look fine on WAPE while consistently over- or under-forecasting zero-heavy items.

## Optional: limited history

Take 400 products and shorten their history to the most recent 13 or 26 weeks at every cutoff, leaving every other product's history untouched. The global model can still learn seasonal and promotional patterns from the other 1,600 products; a local model has only the short window.

With 13 weeks of history the lag-13 and 26-week rolling features are missing for these products' training rows, so they contribute little to the fit and the forecast leans on the other products' patterns.

This is a truncated-history experiment, not a cold start: these products did sell before the truncation point and keep their most recent weeks. A true new product has no history at all.

In [12]:
focal = pd.Series(sorted(panel.unique_id.unique())).sample(400, random_state=7)
short = {}
for weeks in [13, 26]:
    runs = []
    for cutoff in list(DEV_CUTOFFS) + [TEST_CUTOFF]:
        history = actuals[actuals.ds <= cutoff]
        keep = ~history.unique_id.isin(focal) | (history.ds > cutoff - pd.Timedelta(weeks=weeks))
        truncated = history[keep]

        mlf = make_model(f"lgbm_global_last{weeks}", [LocalStandardScaler()])
        mlf.fit(truncated)
        preds = mlf.predict(H)
        preds = preds[preds.unique_id.isin(focal)]

        sf = StatsForecast(models=[HistoricAverage(alias=f"HistoricAverage_last{weeks}")], freq=FREQ, n_jobs=1)
        local = sf.forecast(df=truncated[truncated.unique_id.isin(focal)], h=H)
        runs.append(to_long(preds.merge(local, on=["unique_id", "ds"]), cutoff, "04_limited_history"))
    short[weeks] = pd.concat(runs)

limited = pd.concat(short.values())
limited = limited.drop_duplicates(["unique_id", "cutoff", "ds", "model"])

In [13]:
full_history = reference[reference.model.isin(["lgbm_baseline", "HistoricAverage"]) & reference.unique_id.isin(focal)]
rows = pd.concat([limited, full_history]).query("cutoff in @DEV_CUTOFFS.tolist()")
score(rows, actuals).sort_values("wape")[["model", "series", "wape", "bias", "mean_rmsse", "median_rmsse"]].round(3)

,model,series,wape,bias,mean_rmsse,median_rmsse
0,HistoricAverage,400,0.469,-0.088,1.527,0.916
2,HistoricAverage_last26,400,0.494,-0.080,1.574,0.925
1,HistoricAverage_last13,400,0.496,-0.069,1.553,0.906
3,lgbm_baseline,400,0.505,0.157,1.531,0.926
5,lgbm_global_last26,400,0.520,0.121,1.557,0.938
4,lgbm_global_last13,400,0.541,0.120,1.557,0.947


`lgbm_baseline` and `HistoricAverage` without a suffix use the full history for the same 400 products, so the gap to the `_last13` and `_last26` versions is the cost of the missing history.

For these 400 products HistoricAverage beats the global LightGBM even with full history (0.469 against 0.505). Cutting history to 13 weeks costs the global model 3.6 WAPE points and HistoricAverage 2.7, so pooling across products did not protect the global model here.

## Optional: how many series should share one model?

A training-partition decision, separate from model selection in notebook 07. The department models above merge small departments into "Other". Here every level of the merchandise hierarchy gets the same rule: one LightGBM per group with at least `MIN_SERIES` series at the origin, and the global model's forecast for series in smaller groups.

Everything else is the global model's setup: same lags, Fourier terms, scaler, LightGBM parameters, training window and origins. Differences between these LightGBM rows come from partitioning alone. The statistical baselines are a different model family and are shown only as a reference.

Department, class and subclass are fixed item attributes, known at every origin. A missing value forms an `Unknown` group.

In [14]:
import time

MIN_SERIES = 10
items = panel.drop_duplicates("unique_id").set_index("unique_id")
partitions = {level: items[f"{level}_id"].astype("string").fillna("Unknown") for level in ["department", "class", "subclass"]}

partition_forecasts, partition_runs, trained_series = [], [], []
for cutoff in ALL_CUTOFFS:
    history = actuals[actuals.ds <= cutoff]
    series = history.unique_id.unique()

    start = time.perf_counter()
    mlf = make_model("lgbm_global", [LocalStandardScaler()])
    mlf.fit(history)
    global_forecast = mlf.predict(H)
    partition_runs.append(dict(level="global", cutoff=cutoff, models=1, sizes=[len(series)], fallback=0.0,
                               seconds=time.perf_counter() - start, largest_training_rows=len(history)))

    for level, membership in partitions.items():
        name = f"lgbm_{level}_part"
        sizes = membership[membership.index.isin(series)].value_counts()
        trained = membership[membership.isin(sizes[sizes >= MIN_SERIES].index) & membership.index.isin(series)]

        start = time.perf_counter()
        preds = fit_groups(history, trained, name, [LocalStandardScaler()])
        seconds = time.perf_counter() - start

        fallback = global_forecast[~global_forecast.unique_id.isin(trained.index)].rename(columns={"lgbm_global": name})
        partition_forecasts.append(to_long(pd.concat([preds, fallback]), cutoff, "04_partition_granularity"))
        trained_series.append(pd.DataFrame({"unique_id": trained.index, "cutoff": cutoff, "level": level}))
        partition_runs.append(dict(level=level, cutoff=cutoff, models=trained.nunique(), sizes=sizes[sizes >= MIN_SERIES].tolist(),
                                   fallback=1 - len(trained) / len(series), seconds=seconds,
                                   largest_training_rows=history.unique_id.map(trained).value_counts().max()))

partition_forecasts = pd.concat(partition_forecasts)
partition_runs = pd.DataFrame(partition_runs)
trained_series = pd.concat(trained_series)

### Models and compute

Model counts, series per model and fallback share at the final origin; training time summed over all ten origins (fit and 14-week forecast, one CPU thread). Peak memory is not measured. The largest training set per model, in rows, stands in for it.

In [15]:
final = partition_runs[partition_runs.cutoff == TEST_CUTOFF].set_index("level")
pd.DataFrame({
    "models": final.models,
    "series_median": final.sizes.map(np.median),
    "series_min": final.sizes.map(min),
    "series_max": final.sizes.map(max),
    "fallback_share": final.fallback.round(3),
    "largest_training_rows": final.largest_training_rows,
    "seconds_all_origins": partition_runs.groupby("level").seconds.sum().round(1),
})

,models,series_median,series_min,series_max,fallback_share,largest_training_rows,seconds_all_origins
level,,,,,,,
class,63,14.0,10,56,0.499,5192,56.7
department,65,22.0,10,100,0.127,9300,74.5
global,1,2000.0,2000,2000,0.000,185300,5.1
subclass,23,12.0,10,27,0.846,2495,19.3


### Accuracy and stability

`lgbm_baseline` is the global model (the same setup as `lgbm_global`, fitted in notebook 02). `lgbm_dept_std` is the department variant from above with small departments merged. Development origins first, then the final 14 weeks (exploratory).

In [16]:
partition_reference = pd.concat([reference[reference.model.isin(["Naive", "HistoricAverage", "SeasonalNaive", "lgbm_baseline"])], forecasts[forecasts.model == "lgbm_dept_std"]])
partition_candidates = pd.concat([partition_reference, partition_forecasts])
leaderboard(partition_candidates, actuals, scope="dev")[columns + ["coverage"]].round(3)

,model,wape,bias,mean_rmsse,median_rmsse,apc,apc_median,coverage
0,SeasonalNaive,0.408,-0.042,1.850,1.055,0.000,0.000,0.988
1,HistoricAverage,0.426,-0.060,1.791,0.887,0.015,0.016,1.000
2,lgbm_class_part,0.433,0.088,1.844,0.913,0.154,0.157,1.000
3,lgbm_department_part,0.439,0.106,1.867,0.924,0.178,0.191,1.000
4,lgbm_dept_std,0.440,0.119,1.865,0.918,0.157,0.153,1.000
5,lgbm_subclass_part,0.440,0.129,1.858,0.920,0.130,0.134,1.000
6,lgbm_baseline,0.449,0.139,1.865,0.921,0.120,0.127,1.000
7,Naive,0.454,-0.088,1.860,0.969,0.352,0.370,1.000


In [17]:
leaderboard(partition_candidates, actuals, scope="test")[columns + ["coverage"]].round(3)

,model,wape,bias,mean_rmsse,median_rmsse,apc,apc_median,coverage
0,lgbm_dept_std,0.403,0.079,1.312,0.737,0.133,0.111,1.0
1,lgbm_department_part,0.406,0.091,1.325,0.750,0.162,0.148,1.0
2,lgbm_subclass_part,0.413,0.102,1.301,0.733,0.103,0.084,1.0
3,lgbm_baseline,0.419,0.109,1.302,0.733,0.092,0.076,1.0
4,lgbm_class_part,0.424,0.100,1.314,0.742,0.139,0.116,1.0
5,HistoricAverage,0.426,-0.011,1.367,0.739,0.010,0.010,1.0
6,SeasonalNaive,0.438,-0.037,1.525,0.929,0.000,0.000,1.0
7,Naive,0.470,0.066,1.332,0.771,0.327,0.308,1.0


### Only the series that got their own model

Fallback series are identical to the global model, so the rows above dilute any partition effect. Here each partition is compared with the global model on the series it actually trained, at each development origin.

In [18]:
rows = []
for level in partitions:
    pair = pd.concat([partition_forecasts[partition_forecasts.model == f"lgbm_{level}_part"], reference[reference.model == "lgbm_baseline"]])
    pair = pair.merge(trained_series[trained_series.level == level], on=["unique_id", "cutoff"])
    pair = pair[pair.cutoff.isin(DEV_CUTOFFS)].assign(model=lambda d: np.where(d.model == "lgbm_baseline", "global", "partitioned"))
    rows.append(score(pair, actuals).assign(level=level))
pd.concat(rows).pivot(index="level", columns="model", values=["series", "wape", "bias", "mean_rmsse"]).loc[list(partitions)].round(3)

series               wape               bias              \
model       global partitioned global partitioned global partitioned   
level                                                                  
department  1746.0      1746.0  0.441       0.429  0.131       0.094   
class       1003.0      1003.0  0.433       0.404  0.121       0.031   
subclass     308.0       308.0  0.518       0.440  0.163       0.075   

           mean_rmsse              
model          global partitioned  
level                              
department      1.887       1.890  
class           1.696       1.654  
subclass        1.586       1.539

### By volume and intermittency

Segments are recomputed at each origin from the training history only, with `series_segments(panel, cutoff)`. Development origins.

In [19]:
origin_segments = pd.concat([series_segments(panel, c).assign(cutoff=c) for c in DEV_CUTOFFS]).rename_axis("unique_id").reset_index()
segment_rows = partition_candidates[partition_candidates.cutoff.isin(DEV_CUTOFFS)].merge(origin_segments[["unique_id", "cutoff", "volume", "intermittency"]], on=["unique_id", "cutoff"])
by_volume = score(segment_rows, actuals, by=["volume"], rmsse=False)
by_zeros = score(segment_rows, actuals, by=["intermittency"], rmsse=False)
pd.concat({
    "wape by volume": by_volume.pivot(index="model", columns="volume", values="wape")[["low", "mid", "high"]],
    "bias by volume": by_volume.pivot(index="model", columns="volume", values="bias")[["low", "mid", "high"]],
    "wape by intermittency": by_zeros.pivot(index="model", columns="intermittency", values="wape")[["regular", "some_zeros", "zero_heavy"]],
    "bias by intermittency": by_zeros.pivot(index="model", columns="intermittency", values="bias")[["regular", "some_zeros", "zero_heavy"]],
}, axis=1).round(3)

wape by volume               bias by volume         \
                                low    mid   high            low    mid   
model                                                                     
HistoricAverage               0.826  0.497  0.415         -0.637 -0.149   
Naive                         0.741  0.498  0.447         -0.373 -0.023   
SeasonalNaive                 0.892  0.554  0.388         -0.655 -0.153   
lgbm_baseline                 0.814  0.522  0.438         -0.488  0.081   
lgbm_class_part               0.794  0.511  0.421         -0.511  0.045   
lgbm_department_part          0.785  0.514  0.428         -0.500  0.050   
lgbm_dept_std                 0.792  0.513  0.429         -0.483  0.064   
lgbm_subclass_part            0.807  0.520  0.428         -0.493  0.071   

                            wape by intermittency                        \
                       high               regular some_zeros zero_heavy   
model                                                                     
HistoricAverage      -0.045                 0.390      0.485      0.986   
Naive                -0.089                 0.424      0.520      0.699   
SeasonalNaive        -0.024                 0.352      0.513      1.055   
lgbm_baseline         0.153                 0.416      0.477      1.266   
lgbm_class_part       0.100                 0.395      0.474      1.240   
lgbm_department_part  0.119                 0.405      0.473      1.225   
lgbm_dept_std         0.132                 0.408      0.470      1.200   
lgbm_subclass_part    0.143                 0.406      0.470      1.267   

                     bias by intermittency                        
                                   regular some_zeros zero_heavy  
model                                                             
HistoricAverage                     -0.006     -0.162     -0.723  
Naive                               -0.070     -0.125     -0.239  
SeasonalNaive                        0.014     -0.135     -0.849  
lgbm_baseline                        0.156      0.119     -0.212  
lgbm_class_part                      0.095      0.088     -0.180  
lgbm_department_part                 0.114      0.101     -0.120  
lgbm_dept_std                        0.133      0.104     -0.180  
lgbm_subclass_part                   0.145      0.112     -0.206

### By origin

WAPE at each development origin and the final cutoff, to see whether any ranking holds.

In [20]:
lgbm_rows = partition_candidates[partition_candidates.model.str.startswith("lgbm")]
per_origin = pd.concat([score(lgbm_rows[lgbm_rows.cutoff == c], actuals, rmsse=False).assign(cutoff=c) for c in list(DEV_CUTOFFS) + [TEST_CUTOFF]])
per_origin.pivot(index="model", columns="cutoff", values="wape").round(3)

cutoff,2024-07-17,2024-08-14,2024-09-11,2024-10-09,2024-11-06,2024-12-04,2025-01-01,2025-04-09
model,,,,,,,,
lgbm_baseline,0.498,0.439,0.453,0.429,0.425,0.481,0.426,0.419
lgbm_class_part,0.457,0.419,0.449,0.425,0.414,0.453,0.415,0.424
lgbm_department_part,0.470,0.444,0.475,0.429,0.418,0.434,0.407,0.406
lgbm_dept_std,0.479,0.440,0.455,0.422,0.422,0.456,0.412,0.403
lgbm_subclass_part,0.483,0.427,0.447,0.422,0.420,0.471,0.417,0.413


### Reading the result

- **Accuracy.** Partitioning improves development WAPE by 0.9 to 1.6 points: global 0.449, department 0.439, class 0.433, subclass 0.440. In the final window department (0.406) and subclass (0.413) stay ahead of the global model (0.419); class falls behind it (0.424).
- **Where it helps.** On the series that got their own model the gain is larger: 1.2 points for department (1,746 series), 2.9 for class (1,003) and 7.8 for subclass (308). Most of it is less overforecasting: bias falls from +0.12 to +0.03 for class and from +0.16 to +0.08 for subclass. Mean RMSSE barely moves. By segment the gains are in high-volume and regular items; zero-heavy items stay poor for every LightGBM (WAPE about 1.2).
- **Consistency.** Class and subclass beat the global model at all seven development origins, department at four. Only department and subclass also beat it in the final window.
- **Stability.** Partitioned forecasts revise more: development APC 0.120 for the global model, 0.130 to 0.178 for the partitions.
- **Compute.** One global model takes 9.5 seconds over ten origins. 65 department models take 132 seconds, 63 class models 106 and 23 subclass models 36, plus the global model for fallback series. More, smaller models cost more total time here, though each needs far less memory (largest training set 9,300 rows against 185,300).
- **Model family.** No LightGBM configuration here reaches seasonal naive (0.408) or historic average (0.426) on development WAPE. That gap is the global model's overforecasting, which partitioning only partly fixes. Direct forecasting and the L1 objective (notebooks 05 and 06) close more of it with one model.

Department partitioning is the only level that helps across the portfolio on both the development origins and the final window, by about 1 to 1.3 points. It costs 65 models instead of one and less stable forecasts, and most of its gain is bias that a different objective also removes. With `MIN_SERIES = 10`, half the class series and 85% of the subclass series fall back to the global model, so those levels are partly global anyway.

Try it later: set `MIN_SERIES` to 5 or 20 and see how the fallback share and the scores move.

In [21]:
save_forecasts(partition_forecasts, "04_partition_granularity")

In [22]:
save_forecasts(forecasts, "04_global_vs_specialized")
save_forecasts(limited, "04_limited_history")